# Feature Engineering

Most of the process of handling redundant features, encoding categorical features and feature slection will be done in this part.

## 1. Loading Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/interim/claims_cleaned.csv')
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange-Claim,NumberOfCars,Year,BasePolicy,FraudFound,AgeGroup
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,No,No,External,none,1 year,3 to 4,1994,Liability,0,21 to 30
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,Yes,No,External,none,no change,1 vehicle,1994,Collision,0,31 to 40
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,No,No,External,none,no change,1 vehicle,1994,Collision,0,41 to 50
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability,0,61 to 70
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,No,No,External,none,no change,1 vehicle,1994,Collision,0,21 to 30


In [3]:
# list out the unique values in columns

for column in df.columns:
    print(column + ': '+ str(df[column].unique()))

Month: ['Dec' 'Jan' 'Oct' 'Jun' 'Feb' 'Nov' 'Apr' 'Mar' 'Aug' 'Jul' 'May' 'Sep']
WeekOfMonth: [5 3 2 4 1]
DayOfWeek: ['Wednesday' 'Friday' 'Saturday' 'Monday' 'Tuesday' 'Sunday' 'Thursday']
Make: ['Honda' 'Toyota' 'Ford' 'Mazda' 'Chevrolet' 'Pontiac' 'Accura' 'Dodge'
 'Mercury' 'Jaguar' 'Nisson' 'VW' 'Saab' 'Saturn' 'Porche' 'BMW' 'Mecedes'
 'Ferrari' 'Lexus']
AccidentArea: ['Urban' 'Rural']
DayOfWeekClaimed: ['Tuesday' 'Monday' 'Thursday' 'Friday' 'Wednesday' 'Saturday' 'Sunday']
MonthClaimed: ['Jan' 'Nov' 'Jul' 'Feb' 'Mar' 'Dec' 'Apr' 'Aug' 'May' 'Jun' 'Sep' 'Oct']
WeekOfMonthClaimed: [1 4 2 3 5]
Sex: ['Female' 'Male']
MaritalStatus: ['Single' 'Married' 'Widow' 'Divorced']
Age: [21 34 47 65 27 20 36 16 30 42 71 52 28 61 38 41 32 40 63 31 45 60 39 55
 35 44 72 29 37 59 49 50 26 48 64 33 74 23 25 56 68 18 51 22 53 46 43 57
 54 69 67 19 78 77 75 80 58 73 24 76 62 79 70 17 66]
Fault: ['Policy Holder' 'Third Party']
PolicyType: ['Sport - Liability' 'Sport - Collision' 'Sedan - Liability'


## 2. Handle Redundant Feature

We are going to drop policy number as it is higly correlated with the year feature. The reason for this is that:

- PolicyNumber is essentially a unique ID for each policy. Hence it has no inherent meaning or predictive value for fraud detection
-   Year on the other hand has temporal information. It captures time-based trends and patterns and fraud patterns may also change over time (e.g., economic conditions, new fraud patterns). Year can also correlate with other features:
    -   Vehicle technology changes over years
    -   Insurance policy term may evolve
    -   fraud detection methods improve, affectiong reported fraud rates

In [4]:
df= df.drop(columns= 'PolicyNumber')

In [5]:
df.columns

Index(['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
       'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex',
       'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory',
       'VehiclePrice', 'RepNumber', 'Deductible', 'DriverRating',
       'Days:Policy-Accident', 'Days:Policy-Claim', 'PastNumberOfClaims',
       'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled',
       'WitnessPresent', 'AgentType', 'NumberOfSuppliments',
       'AddressChange-Claim', 'NumberOfCars', 'Year', 'BasePolicy',
       'FraudFound', 'AgeGroup'],
      dtype='object')

## 3. Encode Categorical Features

### 3.1 Ordinal Encoding for ordered categories

#### 3.1.1 AgeOfVehicle (new -> 7+ years) 

I am using the actual year values to preserve the actual age as much as possible.

In [6]:
df.AgeOfVehicle.unique()

array(['3 years', '6 years', '7 years', 'more than 7', '5 years', 'new',
       '4 years', '2 years'], dtype=object)

In [7]:
VehicleAge_mapping = {
    'new': 0,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    'more than 7': 8 }

df['AgeOfVehicle'] = df['AgeOfVehicle'].map(VehicleAge_mapping)

#### 3.1.2 Vehicle Price

In [8]:
df.VehiclePrice.unique()

array(['more than 69,000', '20,000 to 29,000', '30,000 to 39,000',
       'less than 20,000', '40,000 to 59,000', '60,000 to 69,000'],
      dtype=object)

I will represent each VehiclePrice range by its midpoint to better preserve the actual price magnitude and relationships. For the open-ended category "more than 69,000" I will use 75,000 (approximate midpoint of 70–80k).

In [9]:
VehiclePrice_mapping = {
    'less than 20,000': 10000,
    '20,000 to 29,000': 24500,
    '30,000 to 39,000': 34500,
    '40,000 to 59,000': 49500,
    '60,000 to 69,000': 64500,
    'more than 69,000': 75000}

df['VehiclePrice'] = df['VehiclePrice'].map(VehiclePrice_mapping)

#### 3.1.3 Days:Policy-Accident

In [10]:
df['Days:Policy-Accident'].unique()

array(['more than 30', '15 to 30', 'none', '1 to 7', '8 to 15'],
      dtype=object)

The minimum value will be use here to preserve the ordinal relationship

In [11]:
DaysPolicyAccident_mapping = {
    'none': 0,           # Same day
    '1 to 7': 1,         # Minimum of range
    '8 to 15': 8,        # Minimum of range
    '15 to 30': 15,      # Minimum of range
    'more than 30': 31   # Just above 30
}

df['Days:Policy-Accident'] = df['Days:Policy-Accident'].map(DaysPolicyAccident_mapping)

#### 3.1.4 Days:Policy-Claim

In [12]:
df['Days:Policy-Claim'].unique()

array(['more than 30', '15 to 30', '8 to 15'], dtype=object)

Using minimum value approach similar to Days:Policy-Accident to maintain consistency

In [13]:
DaysPolicyClaim_mapping = {
    '8 to 15': 8,
    '15 to 30': 15,
    'more than 30': 31
}

df['Days:Policy-Claim'] = df['Days:Policy-Claim'].map(DaysPolicyClaim_mapping)

#### 3.1.5 AgeOfPolicyHolder

In [14]:
df['AgeOfPolicyHolder'].unique()

array(['26 to 30', '31 to 35', '41 to 50', '51 to 65', '21 to 25',
       '36 to 40', '16 to 17', 'over 65', '18 to 20'], dtype=object)

Using the minimum age value of each range to preserve age progression and maintain interpretability

In [15]:
AgeOfPolicyHolder_mapping = {
    '16 to 17': 16,
    '18 to 20': 18,
    '21 to 25': 21,
    '26 to 30': 26,
    '31 to 35': 31,
    '36 to 40': 36,
    '41 to 50': 41,
    '51 to 65': 51,
    'over 65': 66
}

df['AgeOfPolicyHolder'] = df['AgeOfPolicyHolder'].map(AgeOfPolicyHolder_mapping)

#### 3.1.6 AgeGroup

In [16]:
df['AgeGroup'].unique()

array(['21 to 30', '31 to 40', '41 to 50', '61 to 70', '16 to 20',
       'over 70', '51 to 60'], dtype=object)

Using minimum age value of each range, consistent with AgeOfPolicyHolder encoding approach.
Mapping: 16-20 → 16, 21-30 → 21, 31-40 → 31, ..., over 70 → 71

In [17]:
AgeGroup_mapping = {
    '16 to 20': 16,
    '21 to 30': 21,
    '31 to 40': 31,
    '41 to 50': 41,
    '51 to 60': 51,
    '61 to 70': 61, 
    'over 70': 71
}

df['AgeGroup'] = df['AgeGroup'].map(AgeGroup_mapping)

#### 3.1.7 MonthClaimed

In [18]:
df['MonthClaimed'].unique()

array(['Jan', 'Nov', 'Jul', 'Feb', 'Mar', 'Dec', 'Apr', 'Aug', 'May',
       'Jun', 'Sep', 'Oct'], dtype=object)

Converting month names to numerical values (Jan=1, Feb=2, ..., Dec=12) to preserve temporal order

In [19]:
Month_mapping = {
    'Jan': 1,
    'Feb': 2,
    'Mar': 3,
    'Apr': 4,
    'May': 5,
    'Jun': 6,
    'Jul': 7,
    'Aug': 8,
    'Sep': 9,
    'Oct': 10,
    'Nov': 11,
    'Dec': 12
}

df['MonthClaimed'] = df['MonthClaimed'].map(Month_mapping)

#### 3.1.8 Month

Converting month names to numerical values (Jan=1, Feb=2, ..., Dec=12), consistent with Month encoding

In [20]:
df['Month'].unique()

array(['Dec', 'Jan', 'Oct', 'Jun', 'Feb', 'Nov', 'Apr', 'Mar', 'Aug',
       'Jul', 'May', 'Sep'], dtype=object)

In [21]:
df['Month'] = df['Month'].map(Month_mapping)

#### 3.1.9 DayOfWeek

In [22]:
df['DayOfWeek'].unique()

array(['Wednesday', 'Friday', 'Saturday', 'Monday', 'Tuesday', 'Sunday',
       'Thursday'], dtype=object)

Converting day names to numerical values (Monday=1, ..., Sunday=7), consistent with DayOfWeek encoding

In [23]:
Week_mapping = {
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6,
    'Sunday': 7
}

df['DayOfWeek'] = df['DayOfWeek'].map(Week_mapping)

#### 3.1.10 DayOfWeekClaimed

Converting day names to numerical values (Monday=1, ..., Sunday=7) to preserve weekly order

In [24]:
df['DayOfWeekClaimed'].unique()

array(['Tuesday', 'Monday', 'Thursday', 'Friday', 'Wednesday', 'Saturday',
       'Sunday'], dtype=object)

In [25]:
df['DayOfWeekClaimed'] = df['DayOfWeekClaimed'].map(Week_mapping)

#### 3.1.11 NumberOfCars

In [26]:
df['NumberOfCars'].unique()

array(['3 to 4', '1 vehicle', '2 vehicles', '5 to 8', 'more than 8'],
      dtype=object)

Using minimum value of each range to represent car count progression. '1 vehicle' = 1, '2 vehicles' = 2, '3 to 4' = 3, '5 to 8' = 5, 'more than 8' = 9

In [27]:
NumberOfCars_mapping = {
    '1 vehicle': 1,
    '2 vehicles': 2,
    '3 to 4': 3,
    '5 to 8': 5,
    'more than 8': 9
}

df['NumberOfCars'] = df['NumberOfCars'].map(NumberOfCars_mapping)

#### 3.1.12 AddressChange-Claim

In [28]:
df['AddressChange-Claim'].unique()

array(['1 year', 'no change', '4 to 8 years', '2 to 3 years',
       'under 6 months'], dtype=object)

Using time-based encoding to represent recency of address change. 'under 6 months' = 0.5 (most recent), '1 year' = 1, '2 to 3 years' = 2, '4 to 8 years' = 4, 'no change' = 0 (never changed)

In [29]:
AddressChangeClaim_mapping = {
    'no change': 0,
    'under 6 months': 0.5,
    '1 year': 1,
    '2 to 3 years': 2,
    '4 to 8 years': 4
}

df['AddressChange-Claim'] = df['AddressChange-Claim'].map(AddressChangeClaim_mapping)

#### 3.1.13 NumberOfSuppliments

In [30]:
df['NumberOfSuppliments'].unique()

array(['none', 'more than 5', '3 to 5', '1 to 2'], dtype=object)

Using minimum value of each range to represent supplement count progression. 'none' = 0, '1 to 2' = 1, '3 to 5' = 3, 'more than 5' = 6

In [31]:
NumberOfSuppliments_mapping = {
    'none': 0,
    '1 to 2': 1,
    '3 to 5': 3,
    'more than 5': 6
}

df['NumberOfSuppliments'] = df['NumberOfSuppliments'].map(NumberOfSuppliments_mapping)

#### 3.1.14 PastNumberOfClaims

In [32]:
df['PastNumberOfClaims'].unique()

array(['none', '1', '2 to 4', 'more than 4'], dtype=object)

Using minimum value of each range to represent claim count progression. 'none' = 0, '1' = 1, '2 to 4' = 2, 'more than 4' = 5

In [33]:
PastNumberOfClaims_mapping = {
    'none': 0,
    '1': 1,
    '2 to 4': 2,
    'more than 4': 5
}

df['PastNumberOfClaims'] = df['PastNumberOfClaims'].map(PastNumberOfClaims_mapping)

### 3.2 One-hot encoding for nominal categories

In [46]:
nom_cols = [
    'Make',
    'AccidentArea',
    'Sex',
    'MaritalStatus',
    'Fault',
    'VehicleCategory',
    'PoliceReportFiled',
    'WitnessPresent',
    'BasePolicy'
]

for c in nom_cols:
    print(c+ ' : ' + str(df[c].unique()))
    

KeyError: 'Make'


We will first convert the binary columns (`PoliceReportFiled` and `WitnessPresent`) from Yes/No to 1/0.
Looking at the distribution from our EDA, we will group rare car makes into Other before encoding to avoid sparse, noisy columns that could mislead the logistic regression model. Ferrari, Jaguar, Lexus, Porche, Mecedes and BMW are grouped under Other as their total count falls at or below our chosen threshold of 30.

In [44]:
# map binary Yes/No columns
df['PoliceReportFiled'] = df['PoliceReportFiled'].map({'Yes':1,'No':0})
df['WitnessPresent'] = df['WitnessPresent'].map({'Yes':1,'No':0})

In [43]:
# group rare Make into 'Other'
rare_makes = ['Ferrari','Jaguar','Lexus','Porsche','Mecedes','BMW']
df['Make'] = df['Make'].replace(rare_makes,'Other')

In [45]:
# define all nomial columns to OHE
nom_cols = ['Make', 'AccidentArea', 'Sex', 'MaritalStatus','Fault', 'VehicleCategory', 'BasePolicy']

# OHE
df = pd.get_dummies(df, columns= nom_cols,drop_first=True)